# 03 — Gold KPI Analytics (thin wrapper)

Builds `workspace.gold.study_kpi_summary` and `workspace.gold.site_kpi_summary`
via `src/transforms.py` and validates every KPI row against independent SQL
(correlated subqueries).

## KPI Definitions
* **KPI-1 Enrollment Rate** = enrolled subjects / target enrollment (study & site level)
* **KPI-2 Visit Completion Rate** = completed visits / total visits
* **KPI-3 Visit Data Completeness** = visits with actual_date / total visits

In [0]:
# --- Bootstrap: make the repo's src/ package importable + config ---
import sys

from databricks.sdk.runtime import dbutils

try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().value()
    _root = "/" + "/".join(_nb.strip("/").split("/")[:-2])
except Exception:
    _root = "/Users/sreekanthreddy2000@gmail.com/clinical-trial-dq-pip"
sys.path.insert(0, "/Workspace" + _root)

import datetime

from src.transforms import (
    build_site_kpi_summary, build_study_kpi_summary,
    count_kpi_mismatches, independent_kpi_queries,
)

SILVER, GOLD = "workspace.silver", "workspace.gold"
GOLD_TS = datetime.datetime.now(datetime.UTC).isoformat()
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
print(f"Gold KPI analytics started at {GOLD_TS}")

In [0]:
# --- Build both Gold KPI summaries via the src functions ---
def upsert(df, table, key):
    df.createOrReplaceTempView("v_src")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {GOLD}.{table} USING DELTA "
             f"AS SELECT * FROM v_src WHERE 1=0")
    spark.sql(f"""MERGE INTO {GOLD}.{table} t USING v_src s ON t.{key} = s.{key}
                  WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")

upsert(build_study_kpi_summary(
    spark.table(f"{SILVER}.studies"), spark.table(f"{SILVER}.subjects"),
    spark.table(f"{SILVER}.visits"), gold_ts=GOLD_TS), "study_kpi_summary", "study_id")
upsert(build_site_kpi_summary(
    spark.table(f"{SILVER}.sites"), spark.table(f"{SILVER}.subjects"),
    spark.table(f"{SILVER}.visits"), gold_ts=GOLD_TS), "site_kpi_summary", "site_id")

print(f"study_kpi_summary: {spark.table(f'{GOLD}.study_kpi_summary').count()} rows")
print(f"site_kpi_summary: {spark.table(f'{GOLD}.site_kpi_summary').count()} rows")

In [0]:
# --- KPI validation: every row compared against independent SQL ---
from pyspark.sql import functions as F

print("=" * 60)
print("=== KPI VALIDATION (independent SQL, all rows) ===")
print("=" * 60)
for name, (sql, key, gold_table, gold_col) in independent_kpi_queries(SILVER).items():
    expected = spark.sql(sql).select(key, F.col("val").alias(gold_col))
    actual = spark.table(f"{GOLD}.{gold_table}").select(key, gold_col)
    mismatches = count_kpi_mismatches(actual, expected, key, gold_col)
    status = "PASS" if mismatches == 0 else "FAIL"
    print(f"[{status}] {name}: {mismatches} mismatches")
    assert mismatches == 0, f"{name} failed KPI validation"

In [0]:
# --- Gold table previews ---
display(spark.table(f"{GOLD}.study_kpi_summary"))
display(spark.table(f"{GOLD}.site_kpi_summary"))